# Lab 4 — Detection you can check by hand

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kleinric/cv-labs/blob/main/lab-04.ipynb)

**COMS4036A / COMS7050A Computer Vision · Week 4**

**Due: Thursday 20 August 2026, 17:00.** Groups of up to three; one member submits the group's notebook (`.ipynb`) on Moodle. Every member must be able to explain every cell.

Companion reading is [Chapter 4 of the course book](https://courses.ms.wits.ac.za/~richard/cv/book/chapters/04-detection-two-stage.html). Nothing is trained this week. A pretrained detector is handed to you and the work is measuring it. Every quantity in the detection literature — IoU, precision, recall, average precision, mAP — is arithmetic small enough to do on paper. You do it on paper first, then in code, then against the library's version.

The rung of the skills ladder is **implement IoU and mAP yourself before trusting a library's version**. The second theme is evaluation hygiene: what a number computed on 200 images is worth when the benchmark has 4,952.

The Friday session covers Sections 0–4; Sections 5–8 are the take-home half.

## 0. GPU, W&B, and the data

Open this notebook in [Colab](https://colab.research.google.com) — the badge above — and **File ▸ Save a copy in Drive**. Set **Runtime ▸ Change runtime type** to a GPU, and log in to W&B as in the last two labs; this lab logs to a project called `cv-lab4`.

In [ ]:
# Group members — fill in before submitting.
MEMBERS = [
    # ("Student name", "Student number"),
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import wandb
wandb.login()

The data is the test half of [PASCAL VOC 2007](http://host.robots.ox.ac.uk/pascal/VOC/voc2007/) (Everingham et al., 2010): 4,952 photographs, 20 object classes, one box per object, about 450 MB. Every mAP number in Chapter 4's R-CNN lineage is quoted on this set, which is the reason for choosing it. The download takes a minute or two.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from torchvision.datasets import VOCDetection
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.transform import resize_boxes
from torchvision.ops import box_iou, nms

DEV = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)
voc = VOCDetection(".", year="2007", image_set="test", download=True)
print("device:", DEV, "· images:", len(voc))

The 20 class names, in the order VOC uses, and the seeded subset everything below is measured on. Running the whole test set takes a quarter of an hour on a T4; 200 images takes under a minute. Section 3 asks what that shortcut costs.

In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

def subset(seed, n=200):
    return sorted(np.random.RandomState(seed).choice(len(voc), n,
                                                     replace=False).tolist())

SUBSET = subset(0)
print(len(SUBSET), "images ·", SUBSET[:5])

## 1. What the detector returns

A classifier returns one vector per image, the same shape every time. A detector returns a list whose length depends on the photograph, and the length is part of what is being judged. That is what makes the metric awkward, and the first thing to do is look at the output.

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
model = fasterrcnn_resnet50_fpn_v2(weights=weights).eval().to(DEV)
prep = weights.transforms()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("score_thresh", model.roi_heads.score_thresh,
      "· nms_thresh", model.roi_heads.nms_thresh,
      "· detections_per_img", model.roi_heads.detections_per_img)

Those three numbers are the detector's output policy, and none of them is part of the network. Sections 4 and 6 change them.

**Q1.1.** `voc[i]` returns a `(PIL image, dict)` pair, and the dict is the VOC XML file parsed into nested dictionaries: `target["annotation"]["object"]` is a list, one entry per annotated object, each with a `name`, a `bndbox` of `xmin`/`ymin`/`xmax`/`ymax` strings, and a `difficult` flag. Write `ground_truth(target)` returning three arrays — boxes as float $(x_1, y_1, x_2, y_2)$, integer labels indexing `VOC_CLASSES`, and a boolean `difficult`. Then report, over `SUBSET`: the number of annotated objects, how many are flagged difficult, the mean and maximum number of objects per image, and the count of non-difficult objects in each class.

In [ ]:
def ground_truth(target):
    # YOUR CODE HERE
    ...

**Q1.2.** The detector was not trained on VOC. It was trained on COCO, whose 91-slot label space is `weights.meta["categories"]`, and its label 1 is not VOC's class 1. All 20 VOC classes appear in COCO, but six of the names differ. Build `COCO_TO_VOC`, a dict from COCO label index to VOC class index, and print the six pairs whose names are not identical. Detections in the other 71 slots are dropped from here on: VOC does not annotate those categories, so a correct detection of a truck or a bench would score as a false positive.

In [ ]:
# YOUR CODE HERE

**Q1.3.** Take the busiest image in `SUBSET` — the one with the most annotated objects — and run the model on it: `model([prep(img).to(DEV)])[0]`, inside `torch.no_grad()`. Draw two panels side by side: the ground-truth boxes, and the detections scoring above 0.5 labelled with class and score. Print the number of detections above 0.05, above 0.5 and above 0.9, and the number of annotated objects.

In [ ]:
# YOUR CODE HERE

**Q1.4.** None of those four counts agrees with any other. Account for the gaps: why are there so many more detections above 0.05 than there are objects, what stops the count above 0.05 where it stops, and which of the annotated objects would you expect any detector to miss?

*Answer:*

## 2. IoU by hand

Intersection over union of two axis-aligned boxes: the area they share divided by the area they cover between them. Everything downstream — matching, NMS, the threshold in mAP@0.5 — is this one number.

**Q2.1.** Before writing any code, compute IoU on paper for these three pairs, in $(x_1, y_1, x_2, y_2)$ form. Give each as an exact fraction and as a decimal.

| | box $A$ | box $B$ |
|---|---|---|
| (a) | $(0, 0, 10, 10)$ | $(5, 5, 15, 15)$ |
| (b) | $(0, 0, 10, 10)$ | $(2, 2, 6, 6)$ |
| (c) | $(0, 0, 10, 10)$ | $(20, 20, 30, 30)$ |

*Answer:*

**Q2.2.** Now implement `iou(a, b)` for two boxes given as length-4 sequences, and check it against your three answers. Case (c) is the one that breaks naive implementations: subtracting the corners gives a negative width and a negative height, whose product is a positive "intersection". Your code must return exactly 0, and must not divide by zero when both boxes have zero area. Add a fourth case of your own that touches edge to edge without overlapping, and say what it should give.

In [ ]:
def iou(a, b):
    # YOUR CODE HERE
    ...

**Q2.3.** `iou` is called once per pair, and matching needs every pair. Write `iou_matrix(A, B)` taking arrays of shape $(N, 4)$ and $(M, 4)$ and returning the $(N, M)$ matrix of IoUs, with no Python loop over the pairs — broadcast instead. Verify it two ways: against `iou` on your four cases, and against `torchvision.ops.box_iou`. Report the largest absolute difference.

In [ ]:
def iou_matrix(A, B):
    # YOUR CODE HERE
    ...

**Q2.4.** `box_iou` and your function now agree. State what you would have lost by calling `box_iou` from the start, in one sentence, and then say what the largest absolute difference you measured tells you about float32 against float64.

*Answer:*

## 3. Matching, and the metric

A detection is correct if it overlaps a ground-truth object of the same class by at least some IoU, and if that object has not already been claimed. The rule is greedy: take detections in descending score order, and let each one claim the best unclaimed object it overlaps by at least the threshold. Everything unclaimed is a false negative; every detection that claims nothing is a false positive.

One ground truth per detection is what makes a second box on the same object a false positive rather than a second success. Without that rule, a detector could report a hundred copies of every object and score perfectly.

**Q3.1.** Do the metric on paper first. One image, one class, three ground-truth objects, five detections:

| ground truth | box |
|---|---|
| $g_1$ | $(10, 10, 60, 60)$ |
| $g_2$ | $(100, 100, 150, 150)$ |
| $g_3$ | $(200, 30, 260, 90)$ |

| detection | score | box |
|---|---|---|
| $d_1$ | 0.95 | $(12, 12, 58, 58)$ |
| $d_2$ | 0.90 | $(15, 15, 65, 65)$ |
| $d_3$ | 0.75 | $(98, 102, 152, 148)$ |
| $d_4$ | 0.60 | $(300, 300, 350, 350)$ |
| $d_5$ | 0.30 | $(205, 35, 225, 55)$ |

At an IoU threshold of 0.5, mark each detection TP or FP and say which object it claimed. Then write out precision and recall after each detection in turn, and compute average precision by all-point interpolation: replace each precision by the highest precision at that recall or greater, and take the area under the resulting step function. Give AP as an exact fraction.

*Answer:*

**Q3.2.** Implement `match(det_boxes, det_scores, gt_boxes, iou_thr, gt_difficult)`, returning one flag per detection in the order the detections were given: 1 for a true positive, 0 for a false positive, and $-1$ for a detection whose best match is an object flagged difficult. VOC's protocol neither rewards nor punishes those, so $-1$ means "drop this detection from the count entirely". Check your function reproduces your Q3.1 marking exactly.

In [ ]:
def match(det_boxes, det_scores, gt_boxes, iou_thr=0.5, gt_difficult=None):
    # YOUR CODE HERE
    ...

**Q3.3.** Write `pr_curve(flags, scores, n_positive)` returning the precision and recall after each detection, and `average_precision(prec, rec)` doing the all-point interpolation. Difficult matches leave the arrays before anything is accumulated, and `n_positive` counts only non-difficult objects. Run both on the Q3.1 table and check the AP against your fraction — it should agree to every printed digit.

In [ ]:
def pr_curve(flags, scores, n_positive):
    # YOUR CODE HERE
    ...


def average_precision(prec, rec):
    # YOUR CODE HERE
    ...

**Q3.4.** Now the whole subset. Write `mean_ap(dets, gts, iou_thr=0.5)`, which pools every detection of a class across all images, sorts them by score, matches them image by image, and averages the per-class APs over the classes that occur. Run the model over `SUBSET` at its default settings, collect the detections into the form `mean_ap` wants, and report mAP@0.5, mAP@0.75, and the per-class AP table beside each class's object count from Q1.1. Log all of it to a W&B run named `defaults-subset0`, with a config carrying the weights enum, the subset seed and size, the three output-policy numbers, and the IoU threshold.

In [ ]:
def mean_ap(dets, gts, iou_thr=0.5):
    # YOUR CODE HERE
    ...

**Q3.5.** Rerun `mean_ap` with the difficult flag ignored, so that difficult objects become ordinary ground truth and detections matching them count as ordinary true positives. Report both numbers, say which is larger and by how much, and say which of the two you would quote in a paper — bearing in mind that "difficult" was decided by the people who drew the boxes, not by the detector.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q3.6.** 200 images is a fiftieth of the benchmark. Repeat Q3.4's evaluation on `subset(1)`, `subset(2)` and `subset(3)` — three more runs, all logged, differing only in the seed — and report the four mAP@0.5 values and their spread. The book's run over the whole 4,952-image test set, same model, same settings, gives 0.897. Where does your seed-0 number sit relative to the other three and to 0.897, and what would you have reported if you had run only seed 0? Name the classes whose object count in Q1.1 is small enough to make their AP close to meaningless, and say what that does to a mean taken over 20 classes.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 4. Non-maximum suppression

The detector has been suppressing duplicates all along. Turn it off and the suppression becomes yours to write.

**Q4.1.** Get the unsuppressed candidates. Set `model.roi_heads.nms_thresh = 1.0` (no IoU can exceed 1, so nothing is suppressed), `model.roi_heads.score_thresh = 0.01` and `model.roi_heads.detections_per_img = 300`, and run one pass over `SUBSET`, caching per image the boxes, scores and VOC labels of every candidate in a VOC class. Cache them properly: Sections 4 and 6 both sweep thresholds over these candidates, and neither needs the GPU again. Report the total number of candidates and the mean per image, and compare that mean against the number of objects per image from Q1.1.

In [ ]:
# YOUR CODE HERE

**Q4.2.** Implement `my_nms(boxes, scores, thr)`: sort by descending score, take the top box, discard every remaining box whose IoU with it exceeds `thr`, repeat. Return the kept indices. Then `per_class_nms(boxes, scores, labels, thr)`, which does this within each class — two overlapping boxes of different classes are not duplicates. Check `my_nms` against `torchvision.ops.nms` at a threshold of 0.5, class by class, over every image in the subset, and report whether the kept sets are identical everywhere.

In [ ]:
def my_nms(boxes, scores, thr):
    # YOUR CODE HERE
    ...


def per_class_nms(boxes, scores, labels, thr):
    # YOUR CODE HERE
    ...

**Q4.3.** Sweep the NMS IoU threshold from 0.1 to 1.0 in steps of 0.1. At each value, apply `per_class_nms` to the cached candidates at a score threshold of 0.05 and compute mAP@0.5; separately, apply it at a score threshold of 0.5 and compute the overall precision and recall at IoU 0.5. Plot all three against the threshold on one set of axes, mark the mAP maximum, and log the sweep to W&B.

In [ ]:
# YOUR CODE HERE

**Q4.4.** Two things in that plot need explaining. Where is the mAP maximum, and how much better is it than torchvision's default of 0.5? And why does mAP barely move across most of the sweep while precision at a fixed score threshold falls by more than a factor of ten — what does that say about which of the two numbers to quote when reporting a system that someone is going to look at the output of?

*Answer:*

## 5. What the proposals can reach

Chapter 4's two stages divide the work: the region proposal network says where to look, and the second stage says what is there and where exactly. The second stage never sees a region the first did not propose, so the proposals' recall is a ceiling on the whole detector.

`model.rpn` returns its proposals as the first element of its output. A forward hook can copy them out, but they arrive in the resized frame the model works in — every image is scaled so its shorter side is 800 pixels before the backbone sees it — so they have to be mapped back to the original image. `model.transform` reports the resized size, and `resize_boxes` does the mapping.

```python
proposals, sizes = [], []
model.rpn.register_forward_hook(lambda m, i, o: proposals.append(o[0][0].detach()))
model.transform.register_forward_hook(lambda m, i, o: sizes.append(o[0].image_sizes[0]))
# after each model([...]) call, for an image of size (W, H):
boxes = resize_boxes(proposals.pop(), sizes.pop(), (H, W))
```

**Q5.1.** Register the hooks, run the model over `SUBSET` once more, and cache the proposals per image alongside the ground truth. Report how many proposals arrive per image. Then draw the top 50 on the busiest image from Q1.3, with the ground-truth boxes over them in a different colour.

In [ ]:
# YOUR CODE HERE

**Q5.2.** Proposal recall: the fraction of non-difficult ground-truth objects that have at least one proposal overlapping them by at least a given IoU. Compute it for the top 50, 100, 300 and 1,000 proposals, at IoU 0.5, 0.75 and 0.9 — a $4 \times 3$ table.

In [ ]:
# YOUR CODE HERE

**Q5.3.** Read the table two ways. Along the top row, is the number of proposals the thing limiting this detector at IoU 0.5? Down the last column, what happens to the ceiling as the threshold rises, and which stage of the two must be responsible for the final boxes being accurate enough to survive IoU 0.9? Connect that to the gap you measured in Q3.4 between mAP@0.5 and mAP@0.75.

*Answer:*

## 6. The confidence dial

The score threshold is not trained, not tuned by the benchmark, and not fixed by the architecture. It is set by whoever deploys the thing, and it moves the system along a curve.

**Q6.1.** Using the cached candidates from Q4.1 with NMS at 0.5, sweep the score threshold over $\{0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.95\}$. At each value report overall precision, overall recall and mAP@0.5 at IoU 0.5, and the number of detections kept. Plot precision and recall against the threshold on one set of axes, and log the table to W&B.

In [ ]:
# YOUR CODE HERE

**Q6.2.** Two deployments. One is a safety alarm on a mine haul road that must not miss a person; a human reviews every alert. The other tags a photograph library automatically, and nobody checks. Pick a threshold from your sweep for each, quote the precision and recall it buys, and say what the cost of the errors is in each case. Then say what mAP — a number averaged over every threshold at once — fails to tell either operator.

*Answer:*

## 7. The record

Paste links to your W&B runs below. At minimum: the default-settings evaluation of Q3.4, the three extra seeds of Q3.6, and the two sweeps of Q4.3 and Q6.1. Each needs a meaningful name and a complete config — weights enum, subset seed and size, score threshold, NMS threshold, detections per image, and the IoU threshold the metric used.

*W&B run links:*

## 8. Before you submit

- [ ] **Runtime ▸ Restart session and run all** on a GPU runtime, then read every output. The full re-run downloads 450 MB and puts about 1,200 images through the detector; budget twenty minutes on a T4.
- [ ] Group members filled in; every member can explain every cell.
- [ ] Q2.1 and Q3.1 answered from paper, with the arithmetic shown, not copied from the code below them.
- [ ] Every *Answer:* cell answered; W&B links pasted in Section 7.
- [ ] **File ▸ Download ▸ Download .ipynb**, one member submits on Moodle before **Thursday 20 August, 17:00**.